In [ ]:
import sys
import csv
import json
import ast
import re

sys.path.insert(0, '/Users/kim/SKN/SKN21-FINAL-2Team/backend')

from dotenv import load_dotenv
load_dotenv('/Users/kim/SKN/SKN21-FINAL-2Team/backend/.env')

from app.core.llm_factory import LLMFactory
from langchain_core.messages import HumanMessage

llm = LLMFactory.get_llm(model='gpt-4o-mini', temperature=0, llm_type='openai')
print('LLM loaded')

In [ ]:
EVAL_PROMPT = """당신은 RAG 파이프라인 품질 평가 전문가입니다.
아래 정보를 바탕으로 각 항목을 1~5점으로 평가하세요.

[사용자 질문]
{user_input}

[검색된 컨텍스트]
{retrieved_contexts}

[시스템 응답]
{response}

[참조 답변]
{reference}

평가 항목:
1. relevance (답변 관련성): 사용자 질문 대비 응답이 얼마나 관련 있는가 (1=전혀 무관, 5=완전히 관련)
2. faithfulness (문서 근거성): 응답이 검색된 컨텍스트에 근거하는가 (컨텍스트 없으면 3점, 1=전혀 근거없음, 5=완전히 근거함)
3. hallucination (환각 방지): 응답에 사실과 다른 내용이 있는가 (1=심각한 환각, 5=환각 없음)
4. usefulness (추천 유용성): 실제 여행/장소 추천으로 얼마나 유용한가 (1=전혀 유용하지 않음, 5=매우 유용함)

반드시 아래 JSON 형식으로만 응답하세요 (다른 텍스트 없이):
{{\"relevance\": <1-5>, \"faithfulness\": <1-5>, \"hallucination\": <1-5>, \"usefulness\": <1-5>}}
"""

def parse_scores(text):
    text = text.strip()
    match = re.search(r'\{[^}]+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    scores = {}
    for key in ['relevance', 'faithfulness', 'hallucination', 'usefulness']:
        m = re.search(rf'"?{key}"?\s*:\s*(\d)', text)
        if m:
            scores[key] = int(m.group(1))
    return scores

def safe_parse_contexts(ctx_str):
    if not ctx_str or ctx_str.strip() == '':
        return ''
    try:
        lst = ast.literal_eval(ctx_str)
        if isinstance(lst, list):
            return '\n'.join(str(x) for x in lst)
        return str(lst)
    except Exception:
        return ctx_str[:2000]

In [ ]:
csv_path = '/Users/kim/SKN/SKN21-FINAL-2Team/backend/evaluation/evaluate_testdata.csv'
output_path = '/Users/kim/SKN/SKN21-FINAL-2Team/backend/evaluation/generation_summary.json'

rows = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    print(f'컬럼: {headers}')
    for row in reader:
        rows.append(row)

print(f'총 {len(rows)}행 로드')

In [ ]:
results = []
skip_count = 0

for i, row in enumerate(rows):
    user_input = row.get('user_input', '').strip()
    response = row.get('response', '').strip()
    contexts_raw = row.get('retrieved_contexts', '')
    reference = row.get('reference', '').strip()

    if not response:
        skip_count += 1
        print(f'[SKIP] 행 {i+1}: response 비어있음')
        continue

    retrieved_contexts = safe_parse_contexts(contexts_raw)
    contexts_display = retrieved_contexts[:1500] if retrieved_contexts else '(검색된 컨텍스트 없음)'

    prompt = EVAL_PROMPT.format(
        user_input=user_input[:500],
        retrieved_contexts=contexts_display,
        response=response[:1500],
        reference=reference[:500] if reference else '(없음)'
    )

    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        raw_text = resp.content if hasattr(resp, 'content') else str(resp)
        scores = parse_scores(raw_text)

        if not scores or len(scores) < 4:
            print(f'[WARN] 행 {i+1} 파싱 실패, raw: {raw_text[:200]}')
            scores = {'relevance': 3, 'faithfulness': 3, 'hallucination': 3, 'usefulness': 3}

        if not retrieved_contexts:
            scores['faithfulness'] = 3

        results.append({'idx': i, 'query': user_input[:100], 'scores': scores})
        print(f'[OK] 행 {i+1}/{len(rows)}: {scores}')

    except Exception as e:
        print(f'[ERROR] 행 {i+1}: {e}')
        results.append({'idx': i, 'query': user_input[:100], 'scores': {'relevance': 3, 'faithfulness': 3, 'hallucination': 3, 'usefulness': 3}})

print(f'\n평가 완료: {len(results)}건, 스킵: {skip_count}건')

In [ ]:
n = len(results)
avg_relevance = sum(r['scores'].get('relevance', 3) for r in results) / n
avg_faithfulness = sum(r['scores'].get('faithfulness', 3) for r in results) / n
avg_hallucination = sum(r['scores'].get('hallucination', 3) for r in results) / n
avg_usefulness = sum(r['scores'].get('usefulness', 3) for r in results) / n
overall_avg = (avg_relevance + avg_faithfulness + avg_hallucination + avg_usefulness) / 4

risk_cases = []
for r in results:
    s = r['scores']
    if s.get('hallucination', 5) <= 2 or s.get('faithfulness', 5) <= 2:
        risk_cases.append({'query': r['query'], 'scores': s})

generation_summary = {
    'avg_relevance': round(avg_relevance, 4),
    'avg_faithfulness': round(avg_faithfulness, 4),
    'avg_hallucination': round(avg_hallucination, 4),
    'avg_usefulness': round(avg_usefulness, 4),
    'overall_avg': round(overall_avg, 4),
    'risk_count': len(risk_cases),
    'risk_queries': risk_cases,
    'n_evaluated': n,
    'n_skipped': skip_count,
    'per_query': results
}

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(generation_summary, f, ensure_ascii=False, indent=2)

print(f'generation_summary.json 저장: {output_path}')
print(f'avg_relevance   : {avg_relevance:.4f}')
print(f'avg_faithfulness: {avg_faithfulness:.4f}')
print(f'avg_hallucination:{avg_hallucination:.4f}')
print(f'avg_usefulness  : {avg_usefulness:.4f}')
print(f'overall_avg     : {overall_avg:.4f}')
print(f'risk_count      : {len(risk_cases)}')